# Customer Churn Prediction – EDA & Model Training

## Problem Statement
Customer churn causes revenue loss for businesses.
The goal is to predict whether a customer will churn
based on demographic, service, and billing data.

## Business Objective
Identify high-risk customers early and apply retention strategies.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
import pickle

In [ ]:
df = pd.read_csv("../data/Churn.csv.csv")
df.head()

In [ ]:
df.shape
df.info()
df.isna().sum()

In [ ]:
# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Drop missing values
df.dropna(inplace=True)

# Encode target variable
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# Drop customerID (not useful for prediction)
df.drop("customerID", axis=1, inplace=True)

In [ ]:
sns.countplot(x="Churn", data=df)
plt.title("Churn Distribution")
plt.show()
sns.countplot(x="Contract", hue="Churn", data=df)
plt.title("Churn by Contract Type")
plt.xticks(rotation=30)
plt.show()
sns.boxplot(x="Churn", y="MonthlyCharges", data=df)
plt.title("Monthly Charges vs Churn")
plt.show()

## Key Insights from EDA

- Customers with month-to-month contracts have higher churn.
- Customers with higher monthly charges are more likely to churn.
- New customers (low tenure) churn more frequently.

In [ ]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

num_cols = X.select_dtypes(include="number").columns
cat_cols = X.select_dtypes(include="object").columns

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols)
    ]
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
log_model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

log_model.fit(X_train, y_train)

log_preds = log_model.predict(X_test)
log_auc = roc_auc_score(y_test, log_model.predict_proba(X_test)[:,1])

print("Logistic Regression ROC-AUC:", log_auc)

In [ ]:
rf_model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)
rf_auc = roc_auc_score(y_test, rf_model.predict_proba(X_test)[:,1])

print("Random Forest ROC-AUC:", rf_auc)

## Model Comparison

- Logistic Regression: Baseline model
- Random Forest: Higher ROC-AUC and better non-linear learning

Final Model Selected: **Random Forest**

In [ ]:
pickle.dump(
    rf_model,
    open("../model/churn_pipeline.pkl", "wb")
)

print("Model saved successfully!")